<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>


# Bloque 10 · Informes automáticos y correo

Caso del Observatorio documental. Ejecuta las celdas en orden desde la carpeta del bloque. Lee la explicación, modifica un dato del ejemplo y comprueba cómo cambia el resultado. Los archivos generados quedan en una carpeta temporal.

Este bloque utiliza la biblioteca estándar de Python. Las pruebas de correo usan un borrador local y un servidor simulado.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporal = TemporaryDirectory()
salida = Path(temporal.name)
ejemplos = Path("ejemplos")
import json, os, smtplib
from collections import defaultdict
from copy import deepcopy
from email import policy
from email.parser import BytesParser
from html import escape
from string import Template
from unittest.mock import patch
from informe import cargar, validar, resumir, filas_html, renderizar
from enviar_informe import preparar, enviar

## Ejemplo 1 · Decisión y granularidad

El equipo necesita saber qué páginas requieren revisión. El archivo es la unidad del inventario, la página es la unidad de cobertura y un código puede aparecer en más de una página. Cada indicador explicita su denominador.

In [ ]:
documentos = cargar(ejemplos / "indice_pdf.json")
for doc in documentos:
    print(doc["archivo"], doc["estado"], len(doc["paginas"]))
kpi = resumir(documentos)
print(kpi)
assert kpi["con_texto"] == 2 and kpi["paginas"] == 4

## Ejemplo 2 · Contrato y validación

Validamos los estados y la coherencia de caracteres, referencias y numeración. Un documento cifrado se registra como tal y sus páginas quedan fuera del universo observado. Una cobertura de cero exige un denominador positivo.

In [ ]:
invalido = deepcopy(documentos)
invalido[0]["paginas"][0]["caracteres"] = 0
try:
    validar(invalido)
except ValueError as error:
    print(error)
print("Sin páginas:", resumir([])["cobertura"])
print("Con páginas y sin texto:", resumir([documentos[1]])["cobertura"])

## Ejemplo 3 · Reconciliación, referencias y excepciones

La suma de estados de archivos debe cuadrar con el total y la suma de estados de páginas debe cuadrar con las páginas observadas. Usamos un conjunto para contar referencias distintas y un diccionario para mantener la procedencia.

In [ ]:
assert kpi["archivos"] == kpi["inspeccionados"] + kpi["cifrados"] + kpi["errores"]
assert kpi["paginas"] == kpi["con_texto"] + kpi["pendientes"]
origenes = defaultdict(list)
for doc in documentos:
    for pagina in doc["paginas"]:
        for codigo in pagina["ids"]:
            origenes[codigo].append((doc["archivo"], pagina["pagina"]))
assert len(origenes) == kpi["referencias_distintas"]
print(dict(origenes))
print([(d["archivo"], p["pagina"]) for d in documentos for p in d["paginas"]
       if p["estado"] == "sin_texto_extraido"])

## Ejemplo 4 · Modelo de presentación y escape HTML

Preparamos datos para una plantilla con campos explícitos. El escape convierte caracteres especiales de datos en entidades HTML. El informe completo separa resumen, detalle y metodología. HTML facilita la consulta en navegador, PDF fija el diseño y CSV permite reutilizar filas.

In [ ]:
periodo = "Semana 45"
mini = Template("<h1>$periodo</h1><p>$n páginas</p>")
print(mini.substitute(periodo=escape("Semana <45>"), n=kpi["paginas"]))
html = renderizar(documentos, periodo)
assert "50,0%" in html and "DOC-004" in html
assert renderizar(documentos, periodo) == html

## Ejemplo 5 · Publicación y verificación

La identidad de entrega combina periodo y versión. El modo x protege una salida existente. Preparamos y comprobamos la ejecución manual antes de programarla. El programador debe lanzar el mismo comando con rutas resolubles y guardar un registro de ejecución.

In [ ]:
def publicar(ruta, contenido):
    with ruta.open("x", encoding="utf-8") as archivo:
        archivo.write(contenido)

informe_local = salida / "observatorio_semana45_v1.html"
informe_local.unlink(missing_ok=True)
publicar(informe_local, html)
try:
    publicar(informe_local, html)
except FileExistsError:
    print("La entrega ya existe.")
assert informe_local.read_text(encoding="utf-8") == html

## Ejemplo 6 · EmailMessage y comprobación del adjunto

Un mensaje MIME contiene cabeceras, cuerpo breve y archivo adjunto. Guardamos un .eml y volvemos a leerlo para verificar nombre, tipo y bytes del adjunto. Así se puede revisar el resultado sin credenciales.

In [ ]:
remitente = "observatorio@example.test"
destinatario = "equipo@example.test"
mensaje = preparar(informe_local, remitente, destinatario, periodo)
borrador = salida / "informe.eml"
borrador.write_bytes(mensaje.as_bytes())
analizado = BytesParser(policy=policy.default).parsebytes(borrador.read_bytes())
adjunto, = analizado.iter_attachments()
assert adjunto.get_filename() == informe_local.name
assert adjunto.get_payload(decode=True) == informe_local.read_bytes()
print(analizado["Subject"], adjunto.get_content_type())

## Ejemplo 7 · SMTP con TLS y registro

SMTP_SSL inicia con TLS. STARTTLS negocia TLS en una conexión SMTP antes de autenticar. Las variables de entorno configuran el cliente, mientras el servidor simulado permite comprobar la secuencia sin conexión real. La aceptación SMTP es un estado de entrega al servidor. Registra fecha, periodo y resultado sin volcar credenciales. Ante un fallo tras iniciar el envío, revisa el registro antes de repetir.

In [ ]:
config = {"SMTP_HOST": "smtp.example.test", "SMTP_PORT": "587",
          "SMTP_USER": "docencia", "SMTP_PASSWORD": "credencial-ficticia",
          "SMTP_TLS": "starttls"}
with patch.dict(os.environ, config), patch("enviar_informe.smtplib.SMTP") as smtp:
    cliente = smtp.return_value
    enviar(mensaje, remitente, destinatario)
    pasos = [llamada[0] for llamada in cliente.method_calls]
assert pasos == ["ehlo", "starttls", "ehlo", "login", "send_message"]
print(pasos)

## Diez ejercicios

Completa las celdas TODO. Cada enunciado incluye una comprobación para validar tu resultado.

### Ejercicio 01 · Rechazar duplicados

Añade al índice una copia del primer documento. Captura el error de validar en detalle_error.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert "repetido" in detalle_error

### Ejercicio 02 · Añadir una excepción

Añade reservado.pdf con estado cifrado y sin páginas. Guarda el resumen del índice ampliado en con_cifrado.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert con_cifrado["archivos"] == 4 and con_cifrado["cifrados"] == 1 and con_cifrado["cobertura"] == 0.5

### Ejercicio 03 · Mostrar cero y ausencia

Construye una función mostrar_cobertura que devuelva No aplica si recibe None y formatee otros valores como porcentajes con un decimal y coma.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert mostrar_cobertura(None) == "No aplica" and mostrar_cobertura(0) == "0,0%"

### Ejercicio 04 · Distinguir frecuencia de diversidad

Duplica DOC-001 en la primera página del PDF mixto sin alterar el resto. Guarda las apariciones en frecuencia y el número de códigos únicos en diversidad.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert (frecuencia, diversidad) == (3, 2)

### Ejercicio 05 · Escapar valores

Genera en seguro el HTML de un informe para el periodo «Semana <46> & revisión».

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert "Semana &lt;46&gt; &amp; revisión" in seguro and "Semana <46>" not in seguro

### Ejercicio 06 · Hacer visibles las excepciones

Genera en detalle_html el detalle del índice ampliado del ejercicio 2. Conserva una fila para el archivo cifrado.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert "<td>reservado.pdf</td><td>—</td><td>cifrado</td>" in detalle_html

### Ejercicio 07 · Proteger una entrega

Intenta escribir sobre informe_local usando publicar. Guarda si detectas la colisión en bloqueado.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert bloqueado and informe_local.read_text(encoding="utf-8") == html

### Ejercicio 08 · Auditar un borrador

Prepara un segundo borrador para Semana 46, serialízalo y recupéralo como correo2. Valida asunto, destinatario y bytes.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert "Semana 46" in correo2["Subject"] and correo2["To"] == destinatario
# assert next(correo2.iter_attachments()).get_payload(decode=True) == informe_local.read_bytes()

### Ejercicio 09 · Probar TLS implícito

Simula SMTP_SSL usando el modo ssl y puerto 465. Guarda en pasos_ssl los métodos de la sesión.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert pasos_ssl == ["login", "send_message"]

### Ejercicio 10 · Registrar un fallo de autenticación

Simula SMTPAuthenticationError al llamar a login. Guarda un diccionario registro con periodo, estado error_autenticacion y código SMTP, y comprueba que no se llamó a send_message.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert registro == {"periodo": periodo, "estado": "error_autenticacion", "codigo": 535}
# sesion.send_message.assert_not_called()